In [8]:
# IMPORTS
from pynq import Overlay, allocate, PL, MMIO
from pynq.lib.video import VideoMode
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
from enum import Enum
import time
import os

projectDir = "/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator"
PL.reset()
overlay = Overlay(projectDir + "/VivadoHW.bit")
overlay.download()

In [9]:
# SETUP AND DEFINES
IN_WIDTH, IN_HEIGHT = 1280, 720
OUT_WIDTH, OUT_HEIGHT = 1280, 720
IMAGES_DIR = projectDir + "/images"
VALID_IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')
available_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS)] if os.path.exists(IMAGES_DIR) else []
tpg = overlay.v_tpg_0
vdma = overlay.dma_block.axi_vdma_0
dma = overlay.dma_block.axi_dma_0
vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
buttons = overlay.axi_gpios.axi_gpio_0
imageResolution = overlay.axi_gpios.axi_gpio_1
threshold_gpio = overlay.axi_gpios.axi_gpio_2
image_widget = widgets.Image(format='jpeg', width=IN_WIDTH, height=IN_HEIGHT)
stream_switch = overlay.orb_pipeline.axis_switch_0
current_ps_frame = None

class VideoSource(Enum):
    HDMI = (0, 0)
    TPG_COLOR_BAR_BOX = (1, 1)
    TPG_GRID_BOX = (2, 1)
    PS_TO_PL = (3, 2)

ps_to_pl_selected_image = 0
selected_source = None

In [ ]:
def process_and_rearm_dma():
    global current_desc_count, frame_count
    
    # 0x34 is S2MM_DMASR. Bit 1 (0x2) goes high when the DMA is IDLE.
    if (dma.read(0x34) & 0x2):
        # Bypass the broken PYNQ property and read the hardware S2MM_LENGTH register directly
        transferred_bytes = dma.read(0x58)
        
        # Each axiDescriptor is 256-bits (32 bytes)
        nbrOfDescriptors = transferred_bytes // 32
        
        if nbrOfDescriptors > 0:
            # Subtract 1 to ignore the mandatory TLAST dummy packet sent at the end of the frame
            actual_corners = max(0, nbrOfDescriptors - 1)
            current_desc_count = actual_corners
            
            if actual_corners > 0:
                # Reshape array to extract the true corners
                descriptors = dummy_desc_buf[:actual_corners * 8].reshape(actual_corners, 8)
                
                print(f"[Frame {frame_count}] Detected Corners / Descriptors: {actual_corners}")
                for i in range(min(3, actual_corners)):
                    hex_desc = "".join(f"{val:08x}" for val in reversed(descriptors[i]))
                    print(f"  Desc {i}: 0x{hex_desc}")
        
        # Rearm the DMA for the next frame
        dma.recvchannel.transfer(dummy_desc_buf)

def setupVideoSource(new_src: VideoSource):
    global selected_source, ps_to_pl_selected_image, current_ps_frame  
    if selected_source == new_src and new_src != VideoSource.PS_TO_PL:
        return

    tpg.write(0x0000, 0x0)
    stream_switch.write(0x040, 0x80000000)
    stream_switch.write(0x000, 0x2)

    if vdma.readchannel.running:
        vdma.readchannel.stop()
    if vdma.writechannel.running:
        vdma.writechannel.stop()

    vdma.readchannel._mmio.write(0x30, 0x4)
    while vdma.readchannel._mmio.read(0x30) & 0x4:
        pass

    vdma.writechannel._mmio.write(0x00, 0x4)
    while vdma.writechannel._mmio.read(0x00) & 0x4:
        pass

    selected_source = new_src

    imageResolution.channel1.write(IN_HEIGHT, 0xFFFFFFFF)
    imageResolution.channel2.write(IN_WIDTH, 0xFFFFFFFF)
    tpg.write(0x0010, IN_HEIGHT)
    tpg.write(0x0018, IN_WIDTH)

    if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
        tpg.write(0x0020, 9); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
    elif selected_source == VideoSource.TPG_GRID_BOX:
        tpg.write(0x0020, 11); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)

    # Prime the descriptor DMA for the source switch
    if dma.recvchannel.idle:
        dma.recvchannel.transfer(dummy_desc_buf)

    vdma.readchannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    vdma.readchannel.start()
    vdma.readchannel.irqframecount = 1

    stream_switch.write(0x040, selected_source.value[1])
    stream_switch.write(0x000, 0x2)

    if selected_source in [VideoSource.TPG_COLOR_BAR_BOX, VideoSource.TPG_GRID_BOX]:
        tpg.write(0x0000, 0x81)
        
    elif selected_source == VideoSource.PS_TO_PL:
        if available_images:
            current_image_path = available_images[ps_to_pl_selected_image]
            print(f"Loading image [{ps_to_pl_selected_image + 1}/{len(available_images)}] for PS_TO_PL: {current_image_path}")
            
            img = cv2.imread(current_image_path)
            img = cv2.resize(img, (IN_WIDTH, IN_HEIGHT))
            current_ps_frame = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ps_to_pl_selected_image = (ps_to_pl_selected_image + 1) % len(available_images)
            
            vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
            vdma.writechannel.start()
            
            buf = vdma.writechannel.newframe()
            np.copyto(buf, current_ps_frame)
            vdma.writechannel.writeframe(buf)
        else:
            print("No images available to load.")
            current_ps_frame = None
    
    print("done")


threshold_gpio.channel1.write(20, 0xFFFFFFFF)

dummy_desc_buf = allocate(shape=(65536,), dtype=np.uint32) 

if not dma.recvchannel.running:
    dma.recvchannel.start()
dma.recvchannel.transfer(dummy_desc_buf) 

setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
display(image_widget)

print("Streaming via VDMA interrupts...")
old_button_value = buttons.read(0)
frame_count = 0
current_desc_count = 0

try:
    while True:
        process_and_rearm_dma()

        # Check DMA status continuously while waiting for the VDMA frame
        while (vdma.readchannel._mmio.read(0x34) & 0x1000) == 0:
            process_and_rearm_dma()
            
            if vdma.readchannel._mmio.read(0x34) & 0x8980:
                vdma.readchannel._mmio.write(0x34, 0x8980)
            time.sleep(0.001)

        frame = vdma.readchannel.readframe()
        
        # One final check to ensure descriptors for this frame are caught
        process_and_rearm_dma()

        vis = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        
        cv2.putText(vis, f"Descriptors: {current_desc_count}", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        
        _, jpeg = cv2.imencode('.jpg', vis)
        image_widget.value = jpeg.tobytes()
        
        button_value = buttons.read(0)
        if button_value != old_button_value and button_value > 0:
            if button_value & 0x1:
                setupVideoSource(VideoSource.HDMI)
            elif button_value & 0x2:
                setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
            elif button_value & 0x4:
                setupVideoSource(VideoSource.TPG_GRID_BOX)
            elif button_value & 0x8:
                setupVideoSource(VideoSource.PS_TO_PL)
        
        old_button_value = button_value
        frame_count += 1
        
except KeyboardInterrupt:
    print("\nStopping stream...")
finally:
    tpg.write(0x00, 0x0)
    if vdma.readchannel.running:
        vdma.readchannel.stop()
    if vdma.writechannel.running:
        vdma.writechannel.stop()
    dummy_desc_buf.close()
    print("Stream stopped cleanly.")

done


Image(value=b'', format='jpeg', height='720', width='1280')

Streaming via VDMA interrupts...
done
Loading image [1/2] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/park.jpg
done
Loading image [2/2] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/park_and_coke.jpg
done
done
done
done
done
done

Stopping stream...
Stream stopped cleanly.


In [5]:
def debug_pipeline():
    print("=== PIPELINE HARDWARE STATUS ===")
    
    # 1. DMA Status (r_brief descriptors)
    print("\n--- DMA (S2MM - Descriptors) ---")
    dma_status = dma.read(0x34)
    print(f"Status Register: {hex(dma_status)}")
    if dma_status & 0x1: print(" [!] HALTED - DMA is stopped.")
    if dma_status & 0x2: print(" [*] IDLE - Waiting for data.")
    if dma_status & 0x10: print(" [FATAL] INTERNAL ERROR (0x5011) - Buffer overflow or TLAST mismatch!")
    if dma_status & 0x20: print(" [FATAL] SLAVE ERROR")
    if dma_status & 0x40: print(" [FATAL] DECODE ERROR")
    if not (dma_status & 0x71): print(" [+] RUNNING - Processing descriptors normally.")

    # 2. VDMA Read Status (Video to PS)
    print("\n--- VDMA Read Channel (S2MM - Video to PS) ---")
    vdma_r_status = vdma.readchannel._mmio.read(0x34)
    print(f"Status Register: {hex(vdma_r_status)}")
    if vdma_r_status & 0x1: print(" [!] HALTED - Read channel is stopped.")
    if vdma_r_status & 0x10: print(" [FATAL] INTERNAL ERROR - AXI stream crashed.")
    if vdma_r_status & 0x80: print(" [!] SOF EARLY ERROR - Frame started before previous finished.")
    if vdma_r_status & 0x100: print(" [!] EOL EARLY ERROR - Line ended too soon (Width mismatch).")
    if vdma_r_status & 0x4000: print(" [!] SOF LATE ERROR - New frame didn't start when expected.")
    if not (vdma_r_status & 0x4191): print(" [+] RUNNING - Capturing video normally.")

    # 3. VDMA Write Status (PS to Video)
    print("\n--- VDMA Write Channel (MM2S - PS to PL) ---")
    vdma_w_status = vdma.writechannel._mmio.read(0x04)
    print(f"Status Register: {hex(vdma_w_status)}")
    if vdma_w_status & 0x1: print(" [!] HALTED")
    if vdma_w_status & 0x10: print(" [FATAL] INTERNAL ERROR")
    if not (vdma_w_status & 0x11): print(" [+] RUNNING - Feeding video normally.")

    # 4. Stream Switch Routing
    print("\n--- AXI Stream Switch ---")
    routing = stream_switch.read(0x040)
    source_map = {0: "HDMI", 1: "TPG", 2: "PS_TO_PL"}
    print(f"Currently routing Si[{routing}] ({source_map.get(routing, 'UNKNOWN')}) to Broadcaster.")

debug_pipeline()

=== PIPELINE HARDWARE STATUS ===

--- DMA (S2MM - Descriptors) ---
Status Register: 0x1000
 [+] RUNNING - Processing descriptors normally.

--- VDMA Read Channel (S2MM - Video to PS) ---
Status Register: 0x10001
 [!] HALTED - Read channel is stopped.

--- VDMA Write Channel (MM2S - PS to PL) ---
Status Register: 0x10001
 [!] HALTED

--- AXI Stream Switch ---
Currently routing Si[1] (TPG) to Broadcaster.


In [ ]:
overlay.orb_pipeline?
